# Stacks and Queues

## Stacks

Stacks are data structures following LIFO(Last In First Out) principle. The principle indicates that the last element added to the stack will be the first one to be removed.

Most common operations:
- Push -> Add the data to the top of the stack.
- Pop -> Remove data from the top of the stack.
- Seek -> Get the data from top of the stack without removing it.

And there are some size-related operations.

### Stacks Implementations

Stacks can be implemented in multiple ways. I will cover two types of implementations. One is using
arrays(I am going to use Python lists). The array-based implementation is simple but it may be inefficient if
the stack keeps growing and shrinking frequently. The other implementation is by using linked lists nodes.
This could be more efficient if the stack keeps growing and shrinking but it may have more overhead due to node structure.

> Overhead means consumption of additional computer resources, such as memory, processing time, and bandwidth but the
> consumption do not directly contribute to the primary goal of the task.

### Stacks Use Cases

- They are used in most of the undo mechanism implementations.
- Function calls. The call stack is used to manage function calls and returns.
- Backtracking algorithms and a good load of other algorithms(and data structures!).
- Expression evaluation. They are used in postfix, prefix, and infix evaluation. I will implement the evaluation functions in the lesson.

### Stacks Variants

- Min stack: A special stack that supports an additional operation to retrieve the minimum element in constant time.
- Max stack: Similar to min stack, but it retrieves the maximum element.
- Deque(double-ended queue): A hybrid data structure supporting stack-like and queue-like operations. Elements can be removed or added from both ends.

> You can use `lists` as stacks in Python. Also, you can use `deque` for appending and popping from both ends in constant time.

Let's implement stacks using the two methods I have talked about. Don't worry, they are simple af.

In [ ]:
class _Node[T]:
    """Represent a singly linked-list node."""

    def __init__(self, data: T) -> None:
        """Initialize a _Node class."""
        self.previous: _Node[T] | None = None
        self.data = data
        self.next: _Node[T] | None = None

    def __str__(self) -> str:
        """Return the string representation of the class."""
        return f"{self.__class__.__name__}<data: {self.data}>"

    def __repr__(self) -> str:
        """Return the official string representation of the class."""
        return f"{self.__class__.__name__}(data={self.data})"


class ListBasedStack[T]:
    """Implementation of a stack using a Python list."""

    def __init__(self) -> None:
        """Initialize an empty stack."""
        self.elements: list[T] = []

    def push(self, data: T) -> None:
        """
        Push an item onto the top of the stack.

        Parameters
        ----------
        data : Any
            The data to be pushed onto the stack.
        """
        self.elements.append(data)

    def pop(self) -> T:
        """
        Remove and return the item at the top of the stack.

        Returns
        -------
        T
            The data of the item removed from the top of the stack.
        """
        if not self.elements:
            raise ValueError("The stack is empty!")
        return self.elements.pop()

    def seek(self) -> T:
        """
        Return the item at the top of the stack.

        Returns
        -------
        T
            The data of the item from the top of the stack.
        """
        if not self.elements:
            raise ValueError("The stack is empty!")
        return self.elements[-1]

    def __len__(self) -> int:
        """Return the size of the stack."""
        return len(self.elements)


class NodeBasedStack[T]:
    """Implementation of a stack using linked list nodes."""

    def __init__(self) -> None:
        """Initializes an empty stack."""
        self.top: _Node[T] | None = None

    def push(self, data: T) -> None:
        """
        Push an item onto the top of the stack.

        Parameters
        ----------
        data : T
            The data to be pushed onto the stack.
        """
        new_node = _Node(data=data)
        new_node.next = self.top
        self.top = new_node

    def pop(self) -> T:
        """
        Remove and return the item at the top of the stack.

        Returns
        -------
        T
            The data of the item removed from the top of the stack.
        """
        if not self.top:
            raise ValueError("The stack is empty!")

        data = self.top.data
        self.top = self.top.next
        return data

    def seek(self) -> T:
        """
        Return the item at the top of the stack.

        Returns
        -------
        T
            The data of the item from the top of the stack.
        """
        if not self.top:
            raise ValueError("The stack is empty!")
        return self.top.data

    def __len__(self) -> int:
        """Return the size of the stack."""
        size = 0
        current_node = self.top
        while current_node:
            size += 1
            current_node = current_node.next
        return size


Easy right? We will now observe the expression evaluation functions created using stacks.

### Stack Use Case Implementation

#### Polish and Reverse Polish Notations

Let's look at this expression: `(3 + 2) * 5`. What's the issue? The issue is that the precedences
are shown using the parentheses. While this is natural to us, it creates some issues in the computers.

Polish mathematicians in the early 20th century came up with alternative notations that eliminate these
ambiguity entirely.

Btw, the normal mathematician notations that we saw earlier is named infix notation. In infix notation, the
operator comes between the operands.

##### Polish Notation

In polish notation, also called prefix notation, the operations are before the operands. Now you see where the
`prefix` name is coming from. So instead of writing `3 + 5`, you write `+ 3 5`. For the previous expression,
you would write `* + 3 2 5`. Let me show you how to read it. When you see a prefix notation, you read from left
to right. First you see `*` so this is the operator and the next two elements are the operands. So the `+ 3 2` is
one of the operands and the next operand is `5`. That's it!

Can you see the beauty? The parentheses are not needed for expressing the precedences.

##### Reverse Polish Notation

Reverse Polish Notation, also called postfix notation, will put the operator after the operands. So, instead of
`(5 * 6) - 10`, instead you would write `5 6 * 10 -`. Stacks are the perfect data structure for evaluating these
expressions.

In [ ]:
import string

OPERATORS = {
    "*": lambda x, y: x * y,
    "+": lambda x, y: x + y,
    "-": lambda x, y: x - y,
    "/": lambda x, y: x / y,
}


def evaluate_rpn(expression: str, separator: str = " ") -> float:
    """
    Evaluate a Reverse Polish Notation (RPN) expression and return the result.

    Parameters
    ----------
    expression : str
        A string representing a valid RPN expression.
    separator: str
        A string which indicates the separator between elements. default is space.

    Returns
    -------
    float
        The result of evaluating the RPN expression.

    Raises
    ------
    ValueError
        If the expression is invalid or cannot be evaluated correctly.
    ZeroDivisionError
        If there is a division by zero in the expression.
    RuntimeError
        If an unexpected situation occur which should not ever happen.

    Examples
    --------
    >>> evaluate_rpn("3 4 + 2 * 7 /")
    2.0
    >>> evaluate_rpn("5 1 2 + 4 * + 3 -")
    14.0
    """
    elements: list[str] = expression.split(separator)

    allowed_elements = set(OPERATORS.keys()).union(set(string.digits))
    if not set(expression.replace(separator, "")).issubset(allowed_elements):
        raise ValueError(
            "Invalid elements in the expression! "
            f"Valid elements consist of: {allowed_elements}"
        )

    stack = []
    for element in elements:
        if element.strip().isdigit():
            stack.append(float(element))
        else:
            # Check the invariant.
            if len(stack) < 2:
                raise ValueError(
                    f"The postfix expression `{expression}` is faulty. Please insert a "
                    "valid one."
                )

            operator = OPERATORS.get(element)
            first_num = stack.pop()
            second_num = stack.pop()
            new_element = operator(second_num, first_num)
            stack.append(new_element)

    # Postcondition validation.
    if len(stack) != 1: 
        raise RuntimeError(
            "Something wen't wrong in the evaluation, the result stack has more than "
            "one element!"
        )
    return stack[0]


def evaluate_pn(expression: str, separator: str = " ") -> float:
    """
    Evaluate a Polish Notation (PN) expression and return the result.

    Parameters
    ----------
    expression : str
        A string representing a valid PN expression.
    separator: str
        A string which indicates the separator between elements. default is space.

    Returns
    -------
    float
        The result of evaluating the PN expression.

    Raises
    ------
    ValueError
        If the expression is invalid or cannot be evaluated correctly.
    ZeroDivisionError
        If there is a division by zero in the expression.
    RuntimeError
        If an unexpected situation occur which should not ever happen.

    Examples
    --------
    >>> evaluate_pn("/ + 5 3 2")
    4.0
    >>> evaluate_pn("- + / 5 2 3 5")
    0.5
    """
    elements: list[str] = expression.split(separator)

    allowed_elements = set(OPERATORS.keys()).union(set(string.digits))
    if not set(expression.replace(separator, "")).issubset(allowed_elements):
        raise ValueError(
            "Invalid elements in the expression! "
            f"Valid elements consist of: {allowed_elements}"
        )

    stack: list[float] = []
    index = len(elements) - 1
    while index >= 0:
        element = elements[index]

        try:
            num = float(element)
            stack.append(num)
        except ValueError:
            operator = OPERATORS.get(element)
            first_num = stack.pop()
            second_num = stack.pop()
            new_num = operator(first_num, second_num)
            stack.append(new_num)
        index -= 1

    # Postcondition validation.
    if len(stack) != 1: 
        raise RuntimeError(
            "Something wen't wrong in the evaluation, the result stack does not "
            "contain exactly on element!"
        )
    return stack[0]


if __name__ == "__main__":
    expression = "3 12 + 5 * 6 /"
    result = evaluate_rpn(expression=expression)
    print(f"rpn expression `{expression}` result: {result}")

    expression = "- + / 5 2 3 5"
    result = evaluate_pn(expression=expression)
    print(f"pn expression `{expression}` result: {result}")


rpn expression `3 12 + 5 * 6 /` result: 12.5
pn expression `- + / 5 2 3 5` result: 0.5


#### Monotonic Stacks

A monotonic stack is a stack that maintain its elements in strictly in increasing or decreasing order.
This data structure is used in the problems where we are dealing with th next or previous greater or smaller element.

Let's finish the yappa yappa yappa and implement the ds.

In [ ]:
class IncreasingStack[T]:
    """A stack that maintains a monotonically increasing order."""

    def __init__(self) -> None:
        """Initialize a IncreasingStack instance."""
        self._stack = []

    def push(self, data: T) -> None:
        """
        Pushes an element onto the stack, maintaining increasing order.

        Parameters
        ----------
        data : T
            The element to be pushed onto the stack.
        """
        while self._stack and (self._stack[-1] > data):
            self._stack.pop()
        self._stack.append(data)

    def pop(self) -> T | None:
        """
        Pops the top element from the stack.

        Returns
        -------
        T or None
            The popped element if the stack is not empty, else None.
        """
        return self._stack.pop() if self._stack else None

    def peek(self) -> T | None:
        """
        Returns the top element without removing it.

        Returns
        -------
        T or None
            The top element if the stack is not empty, else None.
        """
        return self._stack[-1] if self._stack else None

    def __len__(self) -> int:
        """
        Returns the number of elements in the stack.

        Returns
        -------
        int
            The size of the stack.
        """
        return len(self.stack)


class DecreasingStack[T]:
    """A stack that maintains a monotonically decreasing order."""

    def __init__(self) -> None:
        """Initialize a IncreasingStack instance."""
        self._stack = []

    def push(self, data: T) -> None:
        """
        Pushes an element onto the stack, maintaining decreasing order.

        Parameters
        ----------
        data : T
            The element to be pushed onto the stack.
        """
        while self._stack and (self._stack[-1] < data):
            self._stack.pop()
        self._stack.append(data)

    def pop(self) -> T | None:
        """
        Pops the top element from the stack.

        Returns
        -------
        T or None
            The popped element if the stack is not empty, else None.
        """
        return self._stack.pop() if self._stack else None

    def peek(self) -> T | None:
        """
        Returns the top element without removing it.

        Returns
        -------
        T or None
            The top element if the stack is not empty, else None.
        """
        return self._stack[-1] if self._stack else None

    def __len__(self) -> int:
        """
        Returns the number of elements in the stack.

        Returns
        -------
        int
            The size of the stack.
        """
        return len(self.stack)


Ok, now you asking, what was the next greater or smaller element thing I mentioned previous and how do those problems
solve with this data structure. Let's see:

In [ ]:
def next_right_greater_element(numbers: list[int]) -> list[int]:
    """
    Return a list, containing the right greater element of the same index.

    The Next Greater Element problem involves finding, for each element in an array,
    the first element to its right that is greater than the current element. If no
    such element exists, we use -1 to indicate this.

    This problem is efficiently solved using a stack-based approach that processes
    the array from right to left, maintaining candidates for the next greater element.

    Algorithm Approach (Using Stack)
    -------------------------------
    1. Initialize:
       - A result array filled with -1 (default when no greater element is found)
       - An empty stack to track potential next greater elements

    2. Process from right to left:
       - For each element, pop from stack until we find a greater element
       - The top of stack becomes the next greater element for current
       - Push current element onto stack (as it may be next greater for others)

    3. Return the populated result array

    Time Complexity: O(n) - Each element is pushed and popped from stack exactly once
    Space Complexity: O(n) - For the stack and result storage

    Example Visualization
    ---------------------
    Input: [4, 5, 2, 25]
    Processing:
    Index 3 (25): Stack empty → -1, push 25
    Index 2 (2): 25 > 2 → 25, push 2
    Index 1 (5): 2 < 5 → pop 2, 25 > 5 → 25, push 5
    Index 0 (4): 5 > 4 → 5, push 4
    Result: [5, 25, 25, -1]
    """
    numbers_length = len(numbers)
    result = [-1] * numbers_length
    decreasing_stack: list[int] = []

    for index in range((numbers_length - 1), -1, -1):
        num = numbers[index]
        while decreasing_stack and decreasing_stack[-1] < num:
            decreasing_stack.pop()
        result[index] = decreasing_stack[-1] if decreasing_stack else -1
        decreasing_stack.append(num)
    return result


def next_right_smaller_element(numbers: list[int]) -> list[int]:
    """Return a list, containing the right smaller element of the same index."""
    numbers_length = len(numbers)
    result = [-1] * numbers_length
    increasing_stack: list[int] = []

    for index in range((numbers_length - 1), -1, -1):
        num = numbers[index]
        while increasing_stack and increasing_stack[-1] > num:
            increasing_stack.pop()
        result[index] = increasing_stack[-1] if increasing_stack else -1
        increasing_stack.append(num)
    return result


def next_left_smaller_element(numbers: list[int]) -> list[int]:
    """Return a list, containing the left smaller element of the same index."""
    result = [-1] * len(numbers)
    increasing_stack: list[int] = []

    for index, num in enumerate(numbers):
        while increasing_stack and increasing_stack[-1] > num:
            increasing_stack.pop()
        result[index] = increasing_stack[-1] if increasing_stack else -1
        increasing_stack.append(num)
    return result


def next_left_greater_element(numbers: list[int]) -> list[int]:
    """Return a list, containing the left greater element of the same index."""
    result = [-1] * len(numbers)
    decreasing_stack: list[int] = []

    for index, num in enumerate(numbers):
        while decreasing_stack and decreasing_stack[-1] < num:
            decreasing_stack.pop()
        result[index] = decreasing_stack[-1] if decreasing_stack else -1
        decreasing_stack.append(num)
    return result


if __name__ == "__main__":
    numbers = [5, 4, 3, 2, 1]
    result = next_right_greater_element(numbers=numbers)
    print(f"Result of next right greater element of th `{numbers}`: `{result}`")

    numbers = [6, 2, 3, 8, 4]
    result = next_right_smaller_element(numbers=numbers)
    print(f"Result of next right smaller element of th `{numbers}`: `{result}`")

    numbers = [6, 2, 3, 8, 4]
    result = next_left_smaller_element(numbers=numbers)
    print(f"Result of next left smaller element of th `{numbers}`: `{result}`")

    numbers = [6, 2, 3, 8, 4]
    result = next_left_greater_element(numbers=numbers)
    print(f"Result of next left greater element of th `{numbers}`: `{result}`")

Result of next right greater element of th `[5, 4, 3, 2, 1]`: `[-1, -1, -1, -1, -1]`
Result of next right smaller element of th `[6, 2, 3, 8, 4]`: `[2, -1, -1, 4, -1]`
Result of next left smaller element of th `[6, 2, 3, 8, 4]`: `[-1, -1, 2, 3, 3]`
Result of next left greater element of th `[6, 2, 3, 8, 4]`: `[-1, 6, 6, -1, 8]`


Let's look at another famous problem, solved using the monotonic stack idea.
Given a list of stock prices, the stock span for the i-th day is the count of consecutive
days up to and including day i, such that each of those days had a stock price less than or
equal to the price on day i.

In [7]:
def get_stock_spans(stock_prices: list[int]) -> list[int]:
    """Calculate the stock span for each day's price in the given list."""
    result = [0] * len(stock_prices)
    decreasing_stack: list[int] = []

    for index, stock_price in enumerate(stock_prices):
        while decreasing_stack and stock_prices[decreasing_stack[-1]] <= stock_price:
            decreasing_stack.pop()

        result[index] = (
            index + 1
            if not decreasing_stack
            else index - decreasing_stack[-1]
        )
        decreasing_stack.append(index)
    return result


if __name__ == "__main__":
    stock_prices = [10, 4, 5, 90, 120, 80]
    result = get_stock_spans(stock_prices=stock_prices)
    print(result)


[1, 1, 2, 4, 5, 1]


## Queues

A queue is a data structure following the FIFO(First In First Out) principle. It means that the first element added to the queue is the first element that will be removed.

Most common operations:

- Enqueue/Push -> Add an element to the rear end of the queue.
- Dequeue/Pop -> Removing an element from the front end of the queue.
- Rear -> Getting the element of the rear end of the queue without removing it.
- Front -> Getting the element of the front end of the queue without removing it.

And some other size-related operations.

### Queues Implementations

Queues can be implemented in several ways. Two of the ways are array-based and node-based implementations. Similar to stacks. The other ways is creating a queue using
two stacks, they are called inbound stack and outbound stack. I will implement these below.

### Queues Use cases

- Process scheduling: CPU scheduling in operating systems often involves queues to manage processes.
- Message queues: Manages communication between different parts of the system or between systems often involves message queues.
- Print queues: Managing documents waiting to be printed.
- Tasks scheduling: In multitasking environments, queues can be used to manage tasks waiting to be executed.

### Queues Variants

- Priority Queue: A queue where the elements are dequeued based on their priority not their arrival time.
- Circular Queue: A queue where the rear end is wrapped around the front end. This is a very useful and efficient implementation of queues. It uses the free up space at the front end of the queue.

Let's now implement a normal queue with three different approaches.


In [ ]:
import abc


class _Node[T]:
    """Represent a singly linked-list node."""

    def __init__(self, data: T) -> None:
        """Initialize a _Node class."""
        self.data = data
        self.next: _Node[T] | None = None

    def __repr__(self) -> str:
        """Return the official string representation of the class."""
        return f"{self.__class__.__name__}(data={self.data})"


class Queue[T](abc.ABC):
    """Abstract base class for queue data structures."""

    @abc.abstractmethod
    def enqueue(self, data: T) -> None:
        """Add an element to the rear of the queue.

        Parameters
        ----------
        data : T
            The element to add to the queue.
        """

    @abc.abstractmethod
    def dequeue(self) -> T:
        """
        Remove and return the element from the front of the queue.

        Returns
        -------
        T
            The element removed from the front of the queue.

        Raises
        ------
        IndexError
            If the queue is empty when dequeue is called.
        """

    @abc.abstractmethod
    def get_rear(self) -> T:
        """
        Return the element at the rear of the queue without removing it.

        Returns
        -------
        T
            The element at the rear of the queue.

        Raises
        ------
        IndexError
            If the queue is empty.
        """

    @abc.abstractmethod
    def get_front(self) -> T:
        """Return the element at the front of the queue without removing it.

        Returns
        -------
        T
            The element at the front of the queue.

        Raises
        ------
        IndexError
            If the queue is empty.
        """

    @abc.abstractmethod
    def __len__(self) -> int:
        """Return the size of the queue."""

    @property
    def rear(self) -> T:
        """Property for returning the rear end item in the queue."""
        return self.get_rear()

    @property
    def front(self) -> T:
        """Property for returning the front end item in the queue."""
        return self.get_front()


from collections import deque


class DequeBasedQueue[T](Queue[T]):
    """Represent a deque-based queue data structure."""

    def __init__(self) -> None:
        """Initialize a deque-based queue data structure."""
        self._items: deque[T] = deque()

    def enqueue(self, data: T) -> None:
        """
        Append a new node with the given data to the end of the list.

        Parameters
        ----------
        data : T
            The data to enqueue.
        """
        self._items.append(data)

    def dequeue(self) -> T:
        """
        Remove and return the first element in the list.

        Returns
        -------
        T
            The data of the removed element.
        """
        if not self._items:
            raise IndexError("Queue is empty!")
        return self._items.popleft()

    def get_rear(self) -> T:
        """
        Return the rear end element in the queue.

        Returns
        -------
        T
            The data of the rear end element.
        """
        if not self._items:
            raise IndexError("Queue is empty!")
        return self._items[-1]

    def get_front(self) -> T:
        """
        Return the front end element in the queue.

        Returns
        -------
        T
            The data of the front end element.
        """
        if not self._items:
            raise IndexError("Queue is empty!")
        return self._items[0]

    def __len__(self) -> int:
        """Return the size of the queue."""
        return len(self._items)


class NodeBasedQueue[T](Queue[T]):
    """Represent a node-based queue data structure."""

    def __init__(self) -> None:
        """Initialize a node-based queue data structure."""
        self._front_end: _Node[T] | None = None
        self._rear_end: _Node[T] | None = None
        self._size = 0

    def enqueue(self, data: T) -> None:
        """
        Append a new node with the given data to the rear end of the queue.

        Parameters
        ----------
        data : T
            The data to enqueue.
        """
        new_node = _Node(data=data)
        if not self._rear_end:
            self._front_end = self._rear_end = new_node
        else:
            self._rear_end.next = new_node
            self._rear_end = new_node
        self._size += 1

    def dequeue(self) -> T:
        """
        Remove and return the front end element in the queue.

        Returns
        -------
        T
            The data of the removed element.
        """
        if (self._rear_end is None) or (self._front_end is None):
            raise IndexError("Queue is empty!")

        data = self._front_end.data
        old_front = self._front_end
        self._front_end = self._front_end.next

        if self._front_end is None:
            self._rear_end = None

        self._size -= 1
        del old_front
        return data

    def get_rear(self) -> T:
        """
        Return the rear end element in the queue.

        Returns
        -------
        T
            The data of the rear end element.
        """
        if not self._rear_end:
            raise IndexError("Queue is empty!")
        return self._rear_end.data

    def get_front(self) -> T:
        """
        Return the front end element in the queue.

        Returns
        -------
        T
            The data of the front end element.
        """
        if not self._front_end:
            raise IndexError("Queue is empty!")
        return self._front_end.data

    def __len__(self) -> int:
        """Return the size of the queue."""
        return self._size


class StackBasedQueue[T](Queue[T]):
    """Represent a stack-based queue data structure."""

    def __init__(self) -> None:
        """Initialize a node-based queue data structure."""
        self.inbound_stack: list[T] = []
        self.outbound_stack: list[T] = []

    def enqueue(self, data: T) -> None:
        """
        Append a new node with the given data to the end of the list.

        Parameters
        ----------
        data : T
            The data to enqueue.
        """
        self.inbound_stack.append(data)

    def dequeue(self) -> T:
        """
        Remove and return the first element in the list.

        This operation has amortized O(1) time complexity. When the outbound stack is
        empty, all elements are transferred from the inbound stack to the outbound stack
        in O(n) time, but each element is transferred at most once and then can be
        popped in O(1) time. Over a sequence of n operations, the average cost per
        dequeue is constant.

        Returns
        -------
        T
            The data of the removed element.
        """
        if not (self.outbound_stack or self.inbound_stack):
            raise IndexError("Queue is empty!")

        if not self.outbound_stack:
            while self.inbound_stack:
                self.outbound_stack.append(self.inbound_stack.pop())

        return self.outbound_stack.pop()

    def get_rear(self) -> T:
        """
        Return the rear end element in the queue.

        Returns
        -------
        T
            The data of the rear end element.
        """
        if not (self.outbound_stack or self.inbound_stack):
            raise IndexError("Queue is empty!")
        return self.inbound_stack[-1] if self.inbound_stack else self.outbound_stack[0]

    def get_front(self) -> T:
        """
        Return the front end element in the queue.

        Returns
        -------
        T
            The data of the front end element.
        """
        if not (self.outbound_stack or self.inbound_stack):
            raise IndexError("Queue is empty!")
        return self.outbound_stack[-1] if self.outbound_stack else self.inbound_stack[0]

    def __len__(self) -> int:
        """Return the size of the queue."""
        return len(self.outbound_stack) + len(self.inbound_stack)


# This is not a valid implementation for a queue because it does not retrieve the space
# left behind. I wast just challenging my skills, because I saw something similar on
# Reddit and I was wondering that can I implement something similar. By the way, this
# could be an introduction to circular queues.

import array


class PointerBasedQueue(Queue[int]):
    """Represent a pointer-based queue data structure."""

    def __init__(self) -> None:
        """Initialize a pointer-based queue data structure."""
        self.items = array.array("i")
        self.rear: int = -1  # Pointing to the index of the rear end.
        self.front: int = -1  # Pointing to the index of the front end.

    def enqueue(self, data: int) -> None:
        """
        Append a new node with the given data to the end of the list.

        Parameters
        ----------
        data : int
            The data to enqueue.
        """
        self.items.append(data)
        self.rear += 1
        if self.front == -1:
            self.front += 1

    def dequeue(self) -> int:
        """
        Remove and return the first element in the list.

        Returns
        -------
        int
            The data of the removed element.
        """
        if self.front > self.rear:
            raise IndexError("Queue is empty!")
        data = self.items[self.front]
        self.front += 1
        return data

    def get_rear(self) -> int:
        """
        Return the rear end element in the queue.

        Returns
        -------
        int
            The data of the rear end element.
        """
        if self.front > self.rear:
            raise IndexError("Queue is empty!")
        return self.items[self.rear]

    def get_front(self) -> int:
        """
        Return the front end element in the queue.

        Returns
        -------
        int
            The data of the front end element.
        """
        if self.front > self.rear:
            raise IndexError("Queue is empty!")
        return self.items[self.front]

    def __len__(self) -> int:
        """Return the size of the queue."""
        return self.rear - self.front + 1


OK, let's now focus on implementations of the queues two famous variants, let's start
with the priority queue. In a priority queue, each element has a priority and the elements
are dequeued based on their priority not their arrival time. Priority queues have two variants
themselves, called min and max priority queues. In the min priority queue, the element with
the minimum priority will dequeued first. You can guess what is a max priority queue now.
The most efficient way to build a priority queue is to use a binary heap data structure,
but as I haven't mastered it when writing this section, let's have a quick implementation
of max priority queue which more overhead than the main implementation.

In [ ]:
from typing import NamedTuple


class QueueItem[T](NamedTuple):
    """Holds a value with its associated priority in the queue."""

    value: T
    priority: int


class PriorityQueue[T]:
    """Priority queue that orders items by their priority value."""

    def __init__(self) -> None:
        """Initialize an empty priority queue."""
        self._items: list[QueueItem[T]] = []

    def enqueue(self, data: T, priority: int) -> None:
        """Add item to queue with given priority, then sort by priority."""
        self._items.append(QueueItem(value=data, priority=priority))
        self._items.sort(key=lambda i: i.priority)

    def dequeue(self) -> T:
        """Remove and return the highest priority item from the queue."""
        return self._items.pop().value

    def peek(self) -> T:
        """Return the highest priority item without removing it."""
        return self._items[-1].value

    def __len__(self) -> int:
        """Return the current number of items in the queue."""
        return len(self._items)


Now let's jump to the circular queue implementation. The thing is, circular queue has been implemented
to fix the memory issues of the fixed-size array implementation of queues. In the fixed-size array implementation
, two pointers track the front and rear end of the queue. When the front pointer keeps moving forward, an unused
memory space is left behind, that it can be used again by the queue. Circular queues fix this issue. They use the space
behind the front pointer. The thing is, you would never deal with this specific queue problem in Python, because you would
use `deque` where ever you need, but it gives a good conceptional model to how to manage memory efficiently.

In [ ]:
from typing import cast


class CircularQueue[T]:
    """Circular queue with fixed buffer size, managing the internal list efficiently."""

    def __init__(self, buffer_size: int) -> None:
        """Initialize an empty circular queue with the given buffer size."""
        self._items: list[T | None] = [None] * buffer_size
        self._buffer_size = buffer_size

        self._front = 0  # Store the front-end item's index.
        self._rear = 0  # Store the rear-end item's index.

    def is_full(self) -> bool:
        """Return True if no more items can be enqueued."""
        return ((self._rear + 1) % self._buffer_size) == self._front

    def is_empty(self) -> bool:
        """Return True if the queue contains no items."""
        return self._front == self._rear

    def enqueue(self, data: T) -> None:
        """Add an item to the rear of the queue."""
        if self.is_full():
            raise OverflowError("Queue is full!")

        self._items[self._rear] = data
        self._rear = (self._rear + 1) % self._buffer_size

    def dequeue(self) -> T:
        """Remove and return the front item from the queue."""
        if self.is_empty():
            raise IndexError("Queue is empty!")

        data = self._items[self._front]
        self._front = (self._front + 1) % self._buffer_size
        return cast(T, data)  # Logically, is_empty check prevents returning None here.


#### Buffer

Last but not least, I want to mention what is a buffer means generally. A buffer is a place
where you store data until you can process it, or place some data there temporarily.